In [4]:
#Scene Generation Functions
import numpy as np
import random

#Add Target
def add_tar(X,W,N,T,llx,lly,lsx,lsy,lci,lts):

    #X   - background tensor [T x N x N] 
    #W   - zeros tensor      [T x N x N]
    #N   - frame dimension   
    #T   - number of frames
    #llx - boolean for linear x motion
    #lly - boolean for linear y motion
    #lsx - boolean for sinusoidal x motion [NOT USED CURRENTLY]
    #lsy - boolean for sinusoidal y motion [NOT USED CURRENTLY]
    #lci - boolean for elliptical motion (fx = fy)
    #lts - boolean for target shape (square vs gaussian)

    #used for generating random slopes, intercepts and sinusoidal frequency
    cx = np.random.uniform(0,1,size=4) #x random values   
    cy = np.random.uniform(0,1,size=4) #y random values

    Zb = np.array(np.zeros([T,N,N])) #background + targets output variable
    Zt = np.array(np.zeros([T,N,N])) #targets output variable (target mask)
    ca = np.random.uniform(0.2,0.2)  #target amplitude (right now set to be constant)
    
    cs = np.random.uniform(0.2,0.5) #target size, is different for gaussian vs square
    #(0.1,0.2)  ->  2-3 width gaussian ?
    #(2,3)      ->  3   width square   ?
    #(1,2)      ->  2   width square   ?
    
    xv = list(range(N))
    x,y = np.meshgrid(xv, xv) #x is the matrix of x index for each pixel, y is the matrix of y index for each pixel

    mmat = np.zeros([N,N]) - ca #matrix of target amplitude, mainly needed to blend a gaussian target with the background
    
    #for each target the code picks a random time then forces the target to be at some random location in the frame at that time
    #is useful because we can force the target to be in the inner 3rd of the frame at some point

    t0 = T*np.random.uniform(0,1) #forced center time

    #I dont remember why some of these constants are used, most likely to make sure that the magnitude of the velocity is constant or bounded?

    ax = llx*0.5*(cx[0]+1)*random.choice([-1,1]) #x linear movement
    ay = lly*0.5*(cy[0]+1)*random.choice([-1,1]) #y linear movement

    bx = (cx[2]+1)*7; fx = cx[3]/bx #x frequency, x sin amplitude
    by = (cy[2]+1)*7; fy = cy[3]/by #y frequency, y sin amplitude

    if lci==1: fy = fx #forcing the motion to be eliptical

    u = 12; v = 0 #0 or 1
    mx = v*N/u + ((u-2*v)/u)*N*cx[1] - (ax*t0 + bx*np.cos(fx*t0)) #forced x center
    my = v*N/u + ((u-2*v)/u)*N*cy[1] - (ay*t0 + by*np.sin(fy*t0)) #forced y center


    for t in range(T):

        xp = mx + (ax*t + bx*np.cos(fx*t)) #target x location at time t
        yp = my + (ay*t + by*np.sin(fy*t)) #target y location at time t

        if lts==1: #gaussian
            mm1 = np.exp(-cs*(x-xp)**2-cs*(y-yp)**2) #mixing layer - target, gaussian centered at xp,yp
            mm1[mm1<0.1] = 0
            mm2 = 1 - mm1 #mixing later - background
            Zb[t] = mm2*X[t] + mm1*mmat #frame t
            Zt[t] = mm2*W[t] + mm1*mmat 

        else: #square
            xpr = np.round(xp)
            ypr = np.round(yp)
            mm1 = np.zeros([N,N])
            mm1[(x>=xpr)&(x<=xpr+cs)&(y>=ypr)&(y<=ypr+cs)] = 1
            mm2 = 1 - mm1 #mixing later - background
            Zb[t] = mm2*X[t] + mm1*mmat #frame t
            Zt[t] = mm2*W[t] + mm1*mmat 
            
    return Zb,Zt

#Add Noise
def add_noi(Z,W,lno,cn):

    #lno = boolean for noise type
    
    if lno==0: #salt and pepper
        Y = np.copy(Z)
        U = np.copy(W)
        d1,d2,d3 = np.shape(Z)
        s = np.random.rand(d1,d2,d3) < cn
        print(np.any(s))
        p = np.random.rand(d1,d2,d3) < cn
        Y[s] = 0
        Y[p] = 1
        U[s] = 0
        U[p] = 1
    elif lno==1: #uniform
        N = np.random.uniform(0,cn,size=np.shape(Z))
        Y = Z+N
        U = W+N
    elif lno==2: #normal
        N = np.random.normal(0,cn,size=np.shape(Z))
        Y = Z+N
        U = W+N

    return Y,U


In [5]:
#Example Scene Generation
import numpy as np

N = 200 #frame size
T = 200 #frame number
K = 15  #number of targets

sig = 0.1 #noise amplitude

X0 = np.zeros([N,N]) #background, currently 0 because i dont have anything on hand for demo code
W0 = np.zeros([N,N]) #zeros, used to generate target mask

#replicate for frame number
W = np.tile(W0,[T,1,1]) #target              video
X = np.tile(X0,[T,1,1]) #target + background video
B = np.tile(X0,[T,1,1]) #background          video


for k in range(K):

    #making sure there is some motion (not all motion booleans turned off)
    #however does not safe gaurd from motion random values being low 
    check = 0 
    while (check==0):
        v = np.random.randint(0,2,size=6)
        check = np.sum(v[0:3])

    X,W = add_tar(X,W,N,T,v[0],v[1],v[2],v[3],v[4],v[5])

Y,V = add_noi(X,W,2,sig)

#Y = target + background + noise
#V = target + noise
#X = target + background
#W = target
#B = background
  

In [6]:
#Visualize Scenes
from matplotlib import pyplot as plt
from IPython.display import clear_output

A = W #set to which video you want to use (Y,V,X,W,B)

temp = (A-np.min(A))/(np.max(A)-np.min(A)) #setting values between 0 and 1 for viewing
test = np.transpose([temp,temp,temp],[1,2,3,0]) #setting to greyscale
pdim = np.shape(test)

for i in range(pdim[0]):
    plt.imshow(test[i,:,:])
    plt.title('Frame %d' % i)
    
    #comment line below when saving frames
    plt.show()

    #uncomment line below to save frames
    #plt.savefig("C:/Users/Nick/Documents/RA_sp2024/figures/set10/noswgbeis/" + str(i) + '.jpg')
 
    clear_output(wait=True)


KeyboardInterrupt: 